In [1]:
# %pip install -q openai
# %pip install -q google-genai jsonschema
# %pip install -q pydantic

In [2]:
from dotenv import load_dotenv
import os, getpass, json, re
from openai import OpenAI
# os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
# os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

## define PRODUCTS

In [3]:
PRODUCTS = {
  "P1": {"product_name": "Weekend city trip",
         "facts_line": "Two nights in a central hotel with breakfast, flexible dates, and free cancellation up to 48 hours before check-in."},
  "P2": {"product_name": "Fitness app",
         "facts_line": "Personalized workouts from 10–30 minutes, progress tracking, and adjustable plans for strength, cardio, or mobility."},
  "P3": {"product_name": "Noise-cancelling headphones",
         "facts_line": "Active noise cancellation, transparency mode, 30-hour battery, and comfortable over-ear fit for travel or focus."},
  "P4": {"product_name": "Language-learning course",
         "facts_line": "Structured lessons, speaking practice, feedback exercises, and weekly goals designed for steady progress."},
  "P5": {"product_name": "Subscription coffee",
         "facts_line": "Fresh beans delivered monthly, your roast preference, flexible skip/pause, and tasting notes included with each delivery."},
  "P6": {"product_name": "Productivity tool",
         "facts_line": "Task lists, calendars, reminders, and templates that sync across devices, with simple sharing for projects."},
}

## Pydantic schema (our contract)

In [4]:
from pydantic import BaseModel, Field, ConfigDict
from typing import List, Literal, Tuple

class Meta(BaseModel):
    model_config = ConfigDict(extra="forbid")
    product_id: str
    product_name: str
    trait_axis: Literal["EXTRAVERSION", "OPENNESS"]
    variant: Literal["E_PLUS", "E_MINUS", "O_PLUS", "O_MINUS"]
    paraphrase_id: int = Field(ge=1, le=3)
    word_target: int
    word_range: Tuple[int, int]

class Constraints(BaseModel):
    model_config = ConfigDict(extra="forbid")
    facts_line_must_match_verbatim: bool
    no_personality_labels: bool
    no_extra_benefits_or_claims: bool
    no_social_proof_scarcity_authority: bool
    cta_strength_equal_across_variants: bool

class Content(BaseModel):
    model_config = ConfigDict(extra="forbid")
    hook: str
    facts: str
    framing: str
    cta: str
    full_message: str

class SelfChecks(BaseModel):
    model_config = ConfigDict(extra="forbid")
    approx_word_count_full_message: int
    facts_line_verbatim_confirmed: bool
    forbidden_items_present: List[str]

class Stimulus(BaseModel):
    model_config = ConfigDict(extra="forbid")
    meta: Meta
    constraints: Constraints
    content: Content
    self_checks: SelfChecks

## Gemini call (LLM instructions)

In [5]:
from google import genai
from google.genai import types

client = genai.Client()

# ---------
# CONSTANTS
# ---------

WORD_MIN, WORD_MAX = 90, 110

FORBIDDEN_PATTERNS = [
    r"\blimited time\b", r"\bact now\b", r"\bdon't miss\b", r"\bjoin thousands\b",
    r"\bexperts?\b", r"\baward\b", r"\b#1\b", r"\bmost popular\b", r"\bguarantee\b",
    r"\bsale\b", r"\bdiscount\b", r"\bfree trial\b"
]

E_MINUS_BANNED = ["friends", "together", "crowd", "meet people", "party"]
O_MINUS_BANNED = ["discover", "explore", "experiment", "new perspective", "unexpected"]

# ---------
# CONSTANTS
# ---------

def wc(text: str) -> int:
    return len(re.findall(r"\b[\w’'-]+\b", text))

def normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def validate_stimulus(stim, expected_facts: str, variant: str, word_min: int, word_max: int):
    # 1) constraint flags must be true
    c = stim.constraints
    if not all([
        c.facts_line_must_match_verbatim,
        c.no_personality_labels,
        c.no_extra_benefits_or_claims,
        c.no_social_proof_scarcity_authority,
        c.cta_strength_equal_across_variants,
    ]):
        raise ValueError(f"Constraint flags not all true: {c.model_dump()}")

    # 2) self-checks should agree
    if not stim.self_checks.facts_line_verbatim_confirmed:
        raise ValueError("Model self-check says facts not verbatim.")
    if stim.self_checks.forbidden_items_present:
        raise ValueError(f"Model self-check detected forbidden items: {stim.self_checks.forbidden_items_present}")

    # 3) facts line verbatim (hard)
    if stim.content.facts.strip() != expected_facts.strip():
        raise ValueError("FACTS line mismatch (must be verbatim).")

    # 4) full_message must be exact concatenation of parts
    assembled = normalize_spaces(" ".join([
        stim.content.hook,
        stim.content.facts,
        stim.content.framing,
        stim.content.cta
    ]))
    full = normalize_spaces(stim.content.full_message)
    if assembled != full:
        raise ValueError("full_message does not match hook+facts+framing+cta concatenation.")

    # 5) word count range (use your wc())
    count = wc(full)
    if not (word_min <= count <= word_max):
        raise ValueError(f"Word count out of range: {count} (expected {word_min}-{word_max})")

    # 6) forbidden persuasion cues (your own scan)
    low = full.lower()
    hits = [pat for pat in FORBIDDEN_PATTERNS if re.search(pat, low)]
    if hits:
        raise ValueError(f"Forbidden persuasion cues detected: {hits}")

    # 7) variant leak checks
    if variant == "E_MINUS":
        if any(b in low for b in E_MINUS_BANNED):
            raise ValueError(f"E_MINUS contains social cue (banned): {E_MINUS_BANNED}")

    if variant == "O_MINUS":
        if any(b in low for b in O_MINUS_BANNED):
            raise ValueError(f"O_MINUS contains novelty cue (banned): {O_MINUS_BANNED}")

def build_input(product_id: str, variant: str, paraphrase_id: int, PRODUCTS: dict) -> str:
    p = PRODUCTS[product_id]
    trait_axis = "EXTRAVERSION" if variant.startswith("E_") else "OPENNESS"
    return f"""Generate ONE ad message stimulus.

META:
- product_id: {product_id}
- product_name: {p["product_name"]}
- trait_axis: {trait_axis}
- variant: {variant}
- paraphrase_id: {paraphrase_id}
- word_target: 100
- word_range: [{WORD_MIN}, {WORD_MAX}]

FACTS line (must match verbatim exactly):
{p["facts_line"]}
"""

GENERATOR_INSTRUCTIONS = f"""
You are a stimulus generator for a behavioral experiment.

Hard constraints:
- full_message length: {WORD_MIN}–{WORD_MAX} words.
- The FACTS sentence must be reproduced verbatim EXACTLY as provided.
- Do NOT add benefits/claims beyond the FACTS sentence (no discounts, guarantees, testimonials).
- Do NOT use social proof, scarcity/urgency, or authority cues.
- Do NOT mention personality labels.
- Targeting differences must appear ONLY in hook/framing; facts must stay identical.

Structure inside content:
- hook: 1 sentence
- facts: 1 sentence (verbatim)
- framing: 2–3 sentences (targeting happens here)
- cta: 1 neutral sentence (no urgency)
- full_message: hook + facts + framing + cta concatenated with single spaces.

Variant targeting rules:
- E_PLUS: social orientation + energetic/action tone (>=2 social cues, >=2 high-energy cues).
- E_MINUS: calm autonomy/solo tone (>=2 autonomy cues, >=2 low-arousal cues; avoid explicit social words).
- O_PLUS: novelty/exploration + abstract/value framing (>=2 novelty cues, >=2 abstract/value cues).
- O_MINUS: conventional/practical/predictable + concrete utility (>=2 predictability cues, >=2 utility cues; avoid novelty language).
""".strip()

def generate_one(product_id, variant, paraphrase_id, PRODUCTS, model="gemini-2.5-flash"):
    prompt = GENERATOR_INSTRUCTIONS + "\n\n" + build_input(product_id, variant, paraphrase_id, PRODUCTS)

    resp = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.8,
            response_mime_type="application/json",
            response_schema=Stimulus,   # <-- schema-enforced JSON
        ),
    )

    # Guard: empty output
    if not getattr(resp, "text", None) or not resp.text.strip():
        # Print minimal diagnostics (finish reason / safety may explain empties)
        cand = resp.candidates[0] if getattr(resp, "candidates", None) else None
        raise RuntimeError(
            f"Empty response.text. "
            f"finish_reason={getattr(cand, 'finish_reason', None)} "
            f"safety_ratings={getattr(cand, 'safety_ratings', None)}"
        )

    # Parse + validate via Pydantic
    try:
        stim = Stimulus.model_validate_json(resp.text)
    except Exception as e:
        import json as _json
        _raw = getattr(resp, "text", None)
        try:
            _parsed = _json.loads(_raw) if _raw else None
        except Exception as je:
            raise RuntimeError(f"Validation failed and response not valid JSON. resp.text={_raw!r}; json error: {je}"
            ) from e
        raise RuntimeError(
            f"Validation failed. Parsed JSON top-level keys: "
            f"{list(_parsed.keys()) if isinstance(_parsed, dict) else type(_parsed)}; resp.text={_raw!r}"
            ) from e

    # Single authoritative validation
    # Extra hard checks we care about (facts + word count)
    expected = PRODUCTS[product_id]["facts_line"].strip()
    validate_stimulus(stim, expected, variant, WORD_MIN, WORD_MAX)

    # (Optional) meta-consistency check (recommended)
    if stim.meta.product_id != product_id or stim.meta.variant != variant or stim.meta.paraphrase_id != paraphrase_id:
        raise ValueError(f"Meta mismatch: got {stim.meta.model_dump()} expected id={product_id}, variant={variant}, p={paraphrase_id}")

    
    return stim

## Generate 1 stimulus

In [6]:
try:
    stim = generate_one("P1", "E_PLUS", 1, PRODUCTS)
    print(stim.content.full_message)
except Exception:
    import traceback, sys
    traceback.print_exc()
    # also show repr of exception to preserve newlines
    import builtins
    print("\n--- repr(exception) ---")
    print(repr(sys.exc_info()[1]))


--- repr(exception) ---
1 validation error for Schema
properties.meta.properties.word_range.prefixItems
  Extra inputs are not permitted [type=extra_forbidden, input_value=[{'type': 'integer'}, {'type': 'integer'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden


Traceback (most recent call last):
  File "C:\Users\alec.traub\AppData\Local\Temp\ipykernel_32796\866348967.py", line 2, in <module>
    stim = generate_one("P1", "E_PLUS", 1, PRODUCTS)
  File "C:\Users\alec.traub\AppData\Local\Temp\ipykernel_32796\3614154375.py", line 130, in generate_one
    resp = client.models.generate_content(
  File "c:\Users\alec.traub\AppData\Local\miniconda3\envs\llm\lib\site-packages\google\genai\models.py", line 5621, in generate_content
    response = self._generate_content(
  File "c:\Users\alec.traub\AppData\Local\miniconda3\envs\llm\lib\site-packages\google\genai\models.py", line 4259, in _generate_content
    request_dict = _GenerateContentParameters_to_mldev(
  File "c:\Users\alec.traub\AppData\Local\miniconda3\envs\llm\lib\site-packages\google\genai\models.py", line 1447, in _GenerateContentParameters_to_mldev
    _GenerateContentConfig_to_mldev(
  File "c:\Users\alec.traub\AppData\Local\miniconda3\envs\llm\lib\site-packages\google\genai\models.py", l

In [7]:
from pathlib import Path
out_dir = Path('stimuli')
out_dir.mkdir(parents=True, exist_ok=True)
try:
    message = stim.content.full_message
except NameError:
    stim = generate_one('P1', 'E_PLUS', 1, PRODUCTS)
    message = stim.content.full_message
out_path = out_dir / 'gemini-messages.txt'
with open(out_path, 'a', encoding='utf-8') as f:
    f.write(message.replace('\n', ' ') + '\n\n')
print(f'Appended message to {out_path}')

ValidationError: 1 validation error for Schema
properties.meta.properties.word_range.prefixItems
  Extra inputs are not permitted [type=extra_forbidden, input_value=[{'type': 'integer'}, {'type': 'integer'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden